# 属于哪组实验（来源）与 Td 的关系

**这个 notebook 回答"数据属于哪组实验和 Td 有没有关系"**，它读上一个 notebook 的产物 `fixed_去N行_带来源.xlsx`。

## 分工
- `fixed_provenance.ipynb`：溯源（每条来自哪篇）+ 剔除 SMILES 曾损坏的行 + 两篇温度冲突表；
- **本 notebook**：`来源`（0/1/2）与 `Td` 的关系 —— Pearson r、η²、控制分子组成后的残差检验（Q1 / Q2）。

## 产物
`TDT/out/实验组别与Td关系.xlsx`，5 个表：汇总 / 各组Td分布 / 置换落点分布 / Q1与Q2对照 / 两篇分子组成对比

## 方法要点
- 用 **Pearson r**（不用 Spearman）；
- 不只给 p，**给出 20000 次随机打乱的落点分布**（p 只是它的一个压缩）；
- 区分 Q1（原始值差异）与 Q2（差异是不是实验方法造成的）：Q2 要先控制分子组成。


## 1. 导入 & 路径 & 参数


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict

RDLogger.DisableLog("rdApp.*")  #关掉 RDKit 的警告刷屏

#仓库根：从当前目录往上找带 .git 的目录，这样在仓库里任何位置打开都能 import 到根目录的 config.py
ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import config

#读的是上一个 notebook 的产物（文件名里的"去N行"随数据而变，所以按模式找最新的那个）
hits = sorted(config.OUT.glob("fixed_去*行_带来源.xlsx"))
if not hits:
    raise FileNotFoundError(f"在 {config.OUT} 里找不到 fixed_去*行_带来源.xlsx，先跑 data_prep/fixed_provenance.ipynb")
SRC = hits[-1]
print("读：", SRC.name)

OUT_DIR = config.OUT
OUT = OUT_DIR / "实验组别与Td关系.xlsx"

SEED = 114514     #固定随机种子，结果可复现
NPERM = 20000     #置换次数

读： fixed_去10行_带来源.xlsx


## 2. 读主表（x 用 `来源` 列）


In [3]:
df = pd.read_excel(SRC)

y = df["Td(°C)"].to_numpy(dtype = float)
src = df["来源"].to_numpy()
rng = np.random.default_rng(SEED)

print("n =", len(df))
df["来源"].value_counts().sort_index()

n = 1416


来源
0    560
1    450
2    406
Name: count, dtype: int64

## 3. 分子组成描述符（Q2 要用）

7 类官能团计数 + 重原子数 / 环数 / 分子量 / C,N,O 计数 = 13 维。


In [4]:
SMARTS = {
    "硝基": "[N+](=O)[O-]",
    "氨基": "[NX3;H2,H1;!$(NC=O)]",
    "羟基": "[OX2H]",
    "氯": "[Cl]",
    "氟": "[F]",
    "叠氮": "[N-]=[N+]=[N]",
    "偶氮": "N=N",
}
PATS = {k: Chem.MolFromSmarts(v) for k, v in SMARTS.items()}

def feats(s):
    #一个分子的组成描述符
    m = Chem.MolFromSmiles(str(s))
    row = {k: len(m.GetSubstructMatches(p)) for k, p in PATS.items()}
    row["重原子数"] = m.GetNumHeavyAtoms()
    row["环数"] = m.GetRingInfo().NumRings()
    row["分子量"] = Descriptors.MolWt(m)
    row["N数"] = sum(1 for at in m.GetAtoms() if at.GetSymbol() == "N")
    row["O数"] = sum(1 for at in m.GetAtoms() if at.GetSymbol() == "O")
    row["C数"] = sum(1 for at in m.GetAtoms() if at.GetSymbol() == "C")
    return row

X = pd.DataFrame([feats(s) for s in df["SMILES"]], index = df.index)
print("描述符维度", X.shape)
X.head(3)

描述符维度 (1416, 13)


,硝基,氨基,羟基,氯,氟,叠氮,偶氮,重原子数,环数,分子量,N数,O数,C数
0,3,0,0,0,0,0,0,19,2,271.105,7,7,5
1,2,1,0,0,0,0,0,12,1,173.088,5,4,3
2,4,1,0,0,0,0,0,19,1,279.125,7,8,4


## 4. 主分析 1：二元组别 vs Td（Pearson r + 置换落点分布）

只在 2025（记 1）与只在 2023（记 0）两组，n = 1010。

⚠️ 注意两套编码方向：这里的 `x = 1` 表示 2025，所以 **r = +0.1593 是"2025 那篇 Td 更高"**。

若用 `来源` 的原编码（0=2025、1=2023），r 会变成 −0.1593，绝对值不变。

In [5]:
sub2 = df[df["来源"].isin([0, 1])]
y2 = sub2["Td(°C)"].to_numpy(float)
x2 = (sub2["来源"].to_numpy() == 0).astype(float)   #2025 记 1，2023 记 0

r_obs = stats.pearsonr(x2, y2)
r_null = np.array([stats.pearsonr(rng.permutation(x2), y2).statistic for _ in range(NPERM)])
p_perm = float((np.abs(r_null) >= abs(r_obs.statistic)).mean())

print(f"n = {len(sub2)} | 观测 r = {r_obs.statistic:+.4f} | p(公式) = {r_obs.pvalue:.3e} | 置换 p = {p_perm:.6f}")
print(f"随机落点：均值 {r_null.mean():+.4f} | 标准差 {r_null.std():.4f} | 范围 [{r_null.min():+.4f}, {r_null.max():+.4f}]")
print(f"观测值{'超出' if abs(r_obs.statistic) > np.abs(r_null).max() else '落在'}随机落点范围")

n = 1010 | 观测 r = +0.1614 | p(公式) = 2.523e-07 | 置换 p = 0.000000
随机落点：均值 -0.0001 | 标准差 0.0314 | 范围 [-0.1373, +0.1307]
观测值超出随机落点范围


## 5. 主分析 2：三组 vs Td（η² = 方差解释比例）（Q1）

η² 就是单因素方差分析的效应量：**组别能解释 Td 变异的百分之几**。


In [6]:
groups = {g: df.loc[df["来源"] == g, "Td(°C)"].to_numpy(float) for g in [0, 1, 2]}
grand = y.mean()

ss_b = sum(len(v) * (v.mean() - grand) ** 2 for v in groups.values())
ss_t = ((y - grand) ** 2).sum()
eta2 = ss_b / ss_t

eta_null = np.empty(NPERM)
for i in range(NPERM):
    lab = rng.permutation(src)               #打乱组别标签
    m = pd.Series(y).groupby(lab).mean()
    c = pd.Series(y).groupby(lab).size()
    eta_null[i] = (c * (m - grand) ** 2).sum() / ss_t

print(f"η²(观测) = {eta2:.4f}  →  组别解释 {100 * eta2:.2f}% 的 Td 变异")
print(f"ANOVA p = {stats.f_oneway(*groups.values()).pvalue:.3e} | Kruskal-Wallis p = {stats.kruskal(*groups.values()).pvalue:.3e}")
print(f"随机落点：均值 {eta_null.mean():.4f} | 95% 分位 {np.percentile(eta_null, 95):.5f} | 范围 [{eta_null.min():.4f}, {eta_null.max():.4f}]")
print(f"置换 p = {(eta_null >= eta2).mean():.6f}")

η²(观测) = 0.0188  →  组别解释 1.88% 的 Td 变异
ANOVA p = 1.525e-06 | Kruskal-Wallis p = 1.994e-06
随机落点：均值 0.0014 | 95% 分位 0.00423 | 范围 [0.0000, 0.0144]
置换 p = 0.000000


## 6. 主分析 3：控制分子组成后，差异还在吗（Q2）

13 个组成描述符 → RF 5 折 CV 预测 Td → 取残差 → 对残差比两组。
**残差 = 扣掉"分子组成能解释的那部分"之后剩下的温度差异。**


In [7]:
sub = df[df["来源"].isin([0, 1])]
Xs = X.loc[sub.index].to_numpy()
ys = sub["Td(°C)"].to_numpy()
sr = (sub["来源"].to_numpy() == 0).astype(int)

rf = RandomForestRegressor(n_estimators = 300, random_state = SEED)
pred = cross_val_predict(rf, Xs, ys, cv = KFold(5, shuffle = True, random_state = SEED), n_jobs = -1)
resid = ys - pred
r2_model = 1 - np.var(resid) / np.var(ys)

obs_d = resid[sr == 1].mean() - resid[sr == 0].mean()
perm_d = np.empty(NPERM)
for i in range(NPERM):
    p1 = rng.permutation(len(resid))
    perm_d[i] = resid[p1[:sr.sum()]].mean() - resid[p1[sr.sum():]].mean()
p_d = float((np.abs(perm_d) >= abs(obs_d)).mean())

print(f"RF 5 折 CV R² = {r2_model:.3f}（组成能解释多少 Td）")
print(f"两组残差均值差 = {obs_d:+.2f} °C | 置换 p = {p_d:.4f}")
print(f"随机落点：标准差 {perm_d.std():.2f} | 范围 [{perm_d.min():+.2f}, {perm_d.max():+.2f}]")

RF 5 折 CV R² = 0.411（组成能解释多少 Td）
两组残差均值差 = +2.20 °C | 置换 p = 0.5030
随机落点：标准差 3.24 | 范围 [-12.86, +12.81]


## 7. 组装 5 个输出表


In [8]:
summary = pd.DataFrame([
    {"分析": "组别(2025=1/2023=0) vs Td", "n": len(sub2), "统计量": "Pearson r（点二列）",
     "观测值": round(r_obs.statistic, 4), "随机落点均值": round(r_null.mean(), 4),
     "随机落点标准差": round(r_null.std(), 4), "随机落点范围": f"[{r_null.min():+.4f}, {r_null.max():+.4f}]",
     "p值": f"{r_obs.pvalue:.3e}", "置换p": f"{p_perm:.4f}", "效应量": f"R² = {r_obs.statistic ** 2:.4f}",
     "解读": f"差异真实（超出全部随机落点），但只解释 {100 * r_obs.statistic ** 2:.1f}% 变异"},
    {"分析": "三组组别 vs Td（单因素方差分析）", "n": len(df), "统计量": "η²（方差解释比例）",
     "观测值": round(eta2, 4), "随机落点均值": round(eta_null.mean(), 4),
     "随机落点标准差": round(eta_null.std(), 4), "随机落点范围": f"[{eta_null.min():.4f}, {eta_null.max():.4f}]",
     "p值": f"{stats.f_oneway(*groups.values()).pvalue:.3e}", "置换p": f"{(eta_null >= eta2).mean():.4f}",
     "效应量": f"{100 * eta2:.2f}% 的 Td 变异", "解读": f"三组均值不同（p<1e-5），但只解释 {100 * eta2:.2f}% 变异"},
    {"分析": "组别 vs 残差（已控制 13 个组成描述符）", "n": len(sub), "统计量": "两组残差均值差（°C）",
     "观测值": round(obs_d, 2), "随机落点均值": "无意义",
     "随机落点标准差": round(perm_d.std(), 4), "随机落点范围": f"[{perm_d.min():+.2f}, {perm_d.max():+.2f}]",
     "p值": "—", "置换p": f"{p_d:.4f}", "效应量": f"RF 5折 CV R² = {r2_model:.3f}",
     "解读": f"控制组成后差异不显著（置换 p = {p_d:.2f}）→ 差异主要由分子组成解释"},
])

group_stats = pd.DataFrame([{
    "来源": g,
    "含义": {0: "只在2025论文", 1: "只在2023来源", 2: "两篇都有"}[g],
    "n": len(v),
    "Td均值": round(v.mean(), 2),
    "Td中位": round(np.median(v), 2),
    "Td标准差": round(v.std(ddof = 1), 2),
    "Td最小": v.min(),
    "Td最大": v.max(),
} for g, v in groups.items()])

null_dist = pd.DataFrame({
    "统计量": ["Pearson r（组别 0/1 vs Td）"] * 5 + ["η²（三组）"] * 5,
    "分位": ["2.5%", "25%", "50%", "75%", "97.5%"] * 2,
    "随机落点值": list(np.percentile(r_null, [2.5, 25, 50, 75, 97.5]).round(4))
                  + list(np.percentile(eta_null, [2.5, 25, 50, 75, 97.5]).round(5)),
})
null_dist.loc[len(null_dist)] = ["Pearson r（组别 0/1 vs Td）", "观测值", round(r_obs.statistic, 4)]
null_dist.loc[len(null_dist)] = ["η²（三组）", "观测值", round(eta2, 5)]

q1q2 = pd.DataFrame({
    "问题": ["Q1：两组的 Td 原始值一样吗？", "Q2：这个差异是实验方法造成的吗？"],
    "做法": ["直接比较 Td", "先用 13 个组成描述符拟合 Td，取残差再比较"],
    "统计量": ["Pearson r / 均值差", "残差均值差"],
    "结果": [f"r = {r_obs.statistic:+.4f}，p = {r_obs.pvalue:.2e}，均值差 {groups[0].mean() - groups[1].mean():+.2f} °C",
             f"残差差 {obs_d:+.2f} °C，置换 p = {p_d:.4f}"],
    "结论": ["两组 Td 分布差异真实存在（超出随机落点全部范围）", "差异被分子组成解释 → 非测定方法偏差"],
})

comp_by_source = pd.concat([sub[["来源"]], X.loc[sub.index]], axis = 1).groupby("来源").mean().T.round(3)

summary

,分析,n,统计量,观测值,随机落点均值,随机落点标准差,随机落点范围,p值,置换p,效应量,解读
0,组别(2025=1/2023=0) vs Td,1010,Pearson r（点二列）,0.1614,-0.0001,0.0314,"[-0.1373, +0.1307]",2.523e-07,0.0000,R² = 0.0260,差异真实（超出全部随机落点），但只解释 2.6% 变异
1,三组组别 vs Td（单因素方差分析）,1416,η²（方差解释比例）,0.0188,0.0014,0.0014,"[0.0000, 0.0144]",1.525e-06,0.0000,1.88% 的 Td 变异,三组均值不同（p<1e-5），但只解释 1.88% 变异
2,组别 vs 残差（已控制 13 个组成描述符）,1010,两组残差均值差（°C）,2.2000,无意义,3.2370,"[-12.86, +12.81]",—,0.5030,RF 5折 CV R² = 0.411,控制组成后差异不显著（置换 p = 0.50）→ 差异主要由分子组成解释


## 8. 写盘（5 个表）


In [9]:
#写盘：先写临时文件 → 加样式与列宽 → 再原子替换到 OUT（文件被占用/写到一半都不留残骸）
import os
import time
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter

OUT.parent.mkdir(parents = True, exist_ok = True)      #目录不存在也能跑
tmp = OUT.with_name(f"~{OUT.stem}.tmp.xlsx")

with pd.ExcelWriter(tmp, engine = "openpyxl") as w:
    summary.to_excel(w, sheet_name = "汇总", index = False)
    group_stats.to_excel(w, sheet_name = "各组Td分布", index = False)
    null_dist.to_excel(w, sheet_name = "置换落点分布", index = False)
    q1q2.to_excel(w, sheet_name = "Q1与Q2对照", index = False)
    comp_by_source.to_excel(w, sheet_name = "两篇分子组成对比")

wb = load_workbook(tmp)
for ws in wb.worksheets:
    for c in ws[1]:
        c.font = Font(bold = True)
        c.alignment = Alignment(wrap_text = True, vertical = "center")
    ws.freeze_panes = "A2"                            #只冻结表头，不加筛选
    for col in ws.columns:                            #按内容估列宽（中文算 2 个字符）
        widest = 0
        for cell in col:
            v = "" if cell.value is None else str(cell.value)
            widest = max(widest, sum(2 if ord(ch) > 127 else 1 for ch in v))
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(widest + 2, 60)
wb.save(tmp)

for attempt in range(3):                              #被占用就等 0.5 秒重试
    try:
        os.replace(tmp, OUT)
        break
    except PermissionError:
        time.sleep(0.5)
else:
    raise PermissionError(f"{OUT} 被占用（Excel/WPS 可能开着它），先关掉再跑")

print("写出", OUT, "|", OUT.stat().st_size, "字节")
pd.ExcelFile(OUT).sheet_names

写出 C:\Users\czywg\Desktop\Programmer_czy\chem_project\TDT\out\实验组别与Td关系.xlsx | 9718 字节


['汇总', '各组Td分布', '置换落点分布', 'Q1与Q2对照', '两篇分子组成对比']

## 结论（三句话）

> 数字跟着数据走，以第 8 步的 `汇总` 表为准。下面是 2026-10-26（1416 行口径）的读数。

1. **属于哪组实验与 Td 有关系**：Pearson r = **+0.1614**（n = 1010，公式 p = 2.5e-07），观测值超出 20000 次随机落点分布的全部范围。
2. **但量级不大**：η² = **1.88%**，即"哪组实验"只解释不到 2% 的温度差异。
3. **且不是"实验方法"造成的**：控制 13 个组成描述符后，残差差 **+2.20 °C**、置换 p = **0.50**（不显著）→ 差异主要由两批分子的化学组成解释（2023 那批硝基 3.22 个/分子 vs 2025 的 1.79 个/分子）。
